# Replay: Backtest and Live

Stored time-series data can be consumed at two speeds: **max-speed** for
offline backtesting (no sleeping, as fast as the CPU allows) and
**wall-clock replay** where sleeps are injected between events so that
downstream components see realistic inter-arrival gaps. screamer's `pace`
function handles both modes with a single API: it merges N value streams in
index order and yields `(value, index, source)` tuples, optionally sleeping
between events proportional to the index delta divided by `speed`. Critically,
the **values and their order are identical in both modes** - only the timing
differs. This is how `backtest then go live` stays honest.


In [ ]:
import numpy as np
from screamer import pace

# Two stored streams: index arrays hold logical timestamps (e.g. seconds since epoch)
a_v = np.array([1.0, 2.0, 3.0])
a_k = np.array([0, 10, 30], dtype=np.int64)
b_v = np.array([0.5, 2.5])
b_k = np.array([5, 20], dtype=np.int64)


async def drain(agen):
    """Collect all events from an async generator into a list."""
    out = []
    async for e in agen:
        out.append(e)
    return out


## Turning stored streams into an event stream

`pace` is essentially `merge` plus optional pacing. It takes N value streams
and a matching list of index arrays, merges them in index order, and yields
`(value, index, source)` tuples - one per event. With `speed=inf` no
sleeping occurs and events are emitted as fast as the CPU can produce them.
This is the backtest path.


In [ ]:
# Backtest: max speed, index-sorted, no sleeping
# IPython 7+ supports top-level await; no asyncio.run() needed in a notebook
backtest = await drain(pace(a_v, b_v, index=[a_k, b_k], speed=float("inf")))

print("backtest events (value, index, source):")
for v, i, s in backtest:
    print(f"  index={i:3d}  value={v}  source={s}")


## Wall-clock replay with an injectable sleep

With a finite `speed`, `pace` awaits `sleep(index_delta / speed)` between
consecutive events. In production this is `asyncio.sleep`; for testing and
demos you can inject a fake sleep that records how long it would have waited
without actually blocking. This makes the demo deterministic and instant
while still demonstrating the timing logic.


In [ ]:
# Wall-clock replay at speed=2.0 with an injectable fake clock
slept = []


async def fake_sleep(sec):
    """Record requested sleep durations without actually sleeping."""
    slept.append(sec)


replay = await drain(pace(a_v, b_v, index=[a_k, b_k], speed=2.0, sleep=fake_sleep))

print("replay events (value, index, source):")
for v, i, s in replay:
    print(f"  index={i:3d}  value={v}  source={s}")

print()
print("requested sleeps (index_delta / speed):", slept)


## Values are identical - only timing differs

The core guarantee: `pace` never reorders or modifies events. The
`(value, index)` pairs are the same regardless of `speed`; only the inter-event
delays change. A strategy coded against the backtest stream will see
exactly the same sequence of numbers when deployed against the live replay.


In [ ]:
# (value, index) pairs are identical in both modes
assert [(v, i) for v, i, s in backtest] == [(v, i) for v, i, s in replay], \
    "backtest and replay must produce the same (value, index) pairs"

# sources are also the same (same merge order)
assert [s for v, i, s in backtest] == [s for v, i, s in replay]

print("events (value, index): ", [(v, i) for v, i, s in backtest])
print("sleeps in replay mode: ", slept)

# Verify sleep values: index_delta / speed=2.0
merged_idx = [i for v, i, s in backtest]
deltas = [merged_idx[j+1] - merged_idx[j] for j in range(len(merged_idx)-1)]
expected_sleeps = [d / 2.0 for d in deltas]
assert slept == expected_sleeps, "sleep = index_delta / speed"

print("index-deltas in merged stream:", deltas)
print("expected sleeps (delta/2):     ", expected_sleeps)
print("backtest == replay: PASSED")


**Takeaways**

- `pace` = `merge` + optional pacing: it yields index-sorted
  `(value, index, source)` events from N input streams.
- `speed=inf` gives a pure backtest with no sleeping; a finite `speed`
  injects proportional sleeps so wall-clock time tracks index-delta time.
- The `sleep` parameter is injectable - replace `asyncio.sleep` with a fake
  to make replay deterministic and instant in tests.
- The index must be **metric** (subtractable) for wall-clock pacing: nanoseconds,
  seconds, bar positions, etc.
- Values and order are **identical** in backtest and replay: a strategy that
  passes backtesting will see the same number sequence in production.
- In a Jupyter notebook use top-level `await` (IPython 7+); `asyncio.run()`
  is for scripts where no event loop is running yet.
